In [17]:
import matplotlib.pyplot as plt
import pandas as pd
import re
import os

def parse_champsim_output(file_path):
    """Parses the ChampSim output file to extract relevant metrics."""
    metrics = {}

    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} not found!")
        return metrics

    with open(file_path, 'r') as file:
        for line in file:
            if "cumulative IPC" in line:
                match = re.findall(r"\d+\.\d+", line)
                if match:
                    metrics['IPC'] = float(match[0])
            elif "AVERAGE MISS LATENCY" in line and "L2C" in line:
                match = re.findall(r"\d+\.\d+|\d+", line) 
                if match:
                    metrics['L2C Avg Miss Latency'] = float(match[-1]) 
            elif "L2C PREFETCH" in line and "ACCESS" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 4:
                    metrics['L2_PREFETCH Access'] = int(values[-4])  
                    metrics['L2_PREFETCH Hit'] = int(values[-3])    
                    metrics['L2_PREFETCH Miss'] = int(values[-2]) 
            elif "L2C PREFETCH" in line and "REQUESTED" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 5:
                    metrics['L2_PREFETCH Requested'] = int(values[-4])  
                    metrics['L2_PREFETCH Issued'] = int(values[-3])    
                    metrics['L2_PREFETCH Useful'] = int(values[-2]) 
                    metrics['L2_PREFETCH Useless'] = int(values[-1])    

    return metrics

In [18]:
prefetchers = ['ip-stride', 'next-line', 'no', 'va-ampm-lite', 'adapt-dist']
bfs_results = {}
dfs_results = {}
spmv_results = {}

for p in prefetchers:
    bfs_results[p] = parse_champsim_output(f'output/{p}/bfs_{p}.txt')
    dfs_results[p] = parse_champsim_output(f'output/{p}/dfs_{p}.txt')
    spmv_results[p] = parse_champsim_output(f'output/{p}/spmv_{p}.txt')

bfs_df = pd.DataFrame(bfs_results)
bfs_df['Benchmark'] = 'bfs'
dfs_df = pd.DataFrame(dfs_results)  
dfs_df['Benchmark'] = 'dfs'
spmv_df = pd.DataFrame(spmv_results)
spmv_df['Benchmark'] = 'spmv'

df = pd.concat([bfs_df, dfs_df, spmv_df])
df_long = df.reset_index().melt(id_vars=['index', 'Benchmark'], var_name='Prefetcher', value_name='Value')
df_long.rename(columns={'index': 'Metric'}, inplace=True)

In [19]:
metrics = df_long['Metric'].unique()
metrics

array(['IPC', 'L2_PREFETCH Access', 'L2_PREFETCH Hit', 'L2_PREFETCH Miss',
       'L2_PREFETCH Requested', 'L2_PREFETCH Issued',
       'L2_PREFETCH Useful', 'L2_PREFETCH Useless',
       'L2C Avg Miss Latency'], dtype=object)

In [20]:
import plotly.express as px

prefetch_colors={
    "ip-stride": "rgb(255, 105, 135)",  
    "next-line": "rgb(135, 206, 235)",  
    "va-ampm-lite": "rgb(186, 140, 186)",
    "adapt-dist": "rgb(255, 165, 0)",
    "no": "rgb(144, 238, 144)"
}

for metric in metrics:
    if "PREFETCH" not in metric.upper():
        df_metric = df_long[df_long['Metric'] == metric]
    
        fig = px.bar(
            df_metric, 
            x="Benchmark", 
            y="Value", 
            color="Prefetcher", 
            barmode="group", 
            title=f"{metric} Across Benchmarks",
            labels={"Value": metric, "Benchmark": "Benchmark", "Prefetcher": "Prefetcher"},
            color_discrete_map=prefetch_colors
        )

        if not df_metric.empty:
            fig.show()

In [21]:
import matplotlib.pyplot as plt
import pandas as pd
import re
import os

def parse_champsim_output(file_path, benchmark, prefetcher):
    """Parses the ChampSim output file to extract relevant metrics."""
    metrics = {"Benchmark": benchmark, "Prefetcher": prefetcher}

    if not os.path.exists(file_path):
        print(f"Warning: File {file_path} not found!")
        return metrics

    with open(file_path, 'r') as file:
        for line in file:
            if "cumulative IPC" in line:
                match = re.findall(r"\d+\.\d+", line)
                if match:
                    metrics['IPC'] = float(match[0])
            elif "AVERAGE MISS LATENCY" in line and "L2C" in line:
                match = re.findall(r"\d+\.\d+|\d+", line) 
                if match:
                    metrics['L2C Avg Miss Latency'] = float(match[-1]) 
            elif "L2C PREFETCH" in line and "ACCESS" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 4:
                    metrics['L2_PREFETCH Access'] = int(values[-4])  
                    metrics['L2_PREFETCH Hit'] = int(values[-3])    
                    metrics['L2_PREFETCH Miss'] = int(values[-2]) 
            elif "L2C PREFETCH" in line and "REQUESTED" in line:
                values = re.findall(r"\d+", line) 
                if len(values) >= 5:
                    metrics['L2_PREFETCH Requested'] = int(values[-4])  
                    metrics['L2_PREFETCH Issued'] = int(values[-3])    
                    metrics['L2_PREFETCH Useful'] = int(values[-2]) 
                    metrics['L2_PREFETCH Useless'] = int(values[-1])    

    return metrics

In [22]:
prefetchers = ['ip-stride', 'next-line', 'no', 'va-ampm-lite', 'adapt-dist']
bmarks = ['bfs', 'dfs', 'spmv']

benchmark_results = [
    (f'output/{prefetcher}/{bmark}_{prefetcher}.txt', bmark, prefetcher)
    for prefetcher in prefetchers
    for bmark in bmarks
]

metrics_list = [parse_champsim_output(file, benchmark, prefetcher) for file, benchmark, prefetcher in benchmark_results]

df = pd.DataFrame(metrics_list)

df_long = df.melt(id_vars=["Benchmark", "Prefetcher"], var_name="Metric", value_name="Value")

In [23]:
import plotly.graph_objects as go
import pandas as pd

df_metric = df_long[df_long["Metric"].isin(["L2_PREFETCH Useful", "L2_PREFETCH Useless"])]
total_requests = df_metric.groupby(["Benchmark", "Prefetcher", "Metric"], as_index=False)["Value"].sum()
df_pivot = total_requests.pivot_table(index=["Benchmark", "Prefetcher"], columns="Metric", values="Value", aggfunc="sum").reset_index()
df_pivot['L2_PREFETCH Total Requests'] = df_pivot['L2_PREFETCH Useful'] + df_pivot['L2_PREFETCH Useless']
df_pivot = df_pivot[df_pivot["Prefetcher"] != "no"]

textures = {
    "L2_PREFETCH Useful": ".",  
    "L2_PREFETCH Useless": "x"  
}

prefetcher_colors = {
    "ip-stride": "rgb(255, 105, 135)",  
    "next-line": "rgb(135, 206, 235)",  
    "va-ampm-lite": "rgb(186, 140, 186)",
    "adapt-dist": "rgb(255, 165, 0)",
}

data = []

for prefetcher in df_pivot["Prefetcher"].unique():
    metric_data = df_pivot[df_pivot["Prefetcher"] == prefetcher]
    
    if metric_data.empty:
        continue
    
    x_data = metric_data["Benchmark"]
    
    y_useful = metric_data["L2_PREFETCH Useful"]
    y_useless = metric_data["L2_PREFETCH Useless"]
    y_data = metric_data["L2_PREFETCH Total Requests"]
    
    # Add Useful (first part of the stack)
    data.append(
        go.Bar(
            x=x_data,
            y=y_data,
            name=f"{prefetcher} - Useful",
            hovertext=[f"{prefetcher}<br>Useful: {useful_val}<br>" 
                   for useful_val, useless_val in zip(y_useful, y_useless)],
            marker=dict(
                color=prefetcher_colors.get(prefetcher, "gray"),
                pattern_shape=textures["L2_PREFETCH Useful"]
            ),
            offsetgroup=prefetcher, 
            legendgroup=prefetcher,
            showlegend=False
        )
    )
    
    data.append(
        go.Bar(
            x=x_data,
            y=y_useless,
            hovertext=[f"{prefetcher}<br>Useless: {useless_val}<br>" 
                   for useful_val, useless_val in zip(y_useful, y_useless)],
            marker=dict(
                color=prefetcher_colors.get(prefetcher, "gray"),
                pattern_shape=textures["L2_PREFETCH Useless"]
            ),
            offsetgroup=prefetcher, 
            legendgroup=prefetcher,
            showlegend=False  
        )
    )


for prefetcher, color in prefetcher_colors.items():
    data.append(
        go.Bar(
            x=[None], y=[None],  
            name=f"{prefetcher}", 
            marker=dict(
                color=color,  
                pattern_shape=None  
            ),
            showlegend=True, 
            legendgroup="Prefetcher",  
        )
    )

for metric_type, texture in textures.items():
    metric_type_clean = metric_type.replace("L2_PREFETCH ", "")
    data.append(
        go.Bar(
            x=[None], y=[None],
            name=f"{metric_type_clean}",  
            marker=dict(
                pattern_shape=texture, 
                color='rgba(255,255,255,0)' 
            ),
            showlegend=True,
            legendgroup=metric_type, 
        )
    )

layout = go.Layout(
    title="L2C Useful/Useless Prefetch Requests by Prefetcher",
    xaxis=dict(title="Benchmark"),
    yaxis=dict(title="Total Requests"),
    barmode="group",  
    legend=dict(title="Prefetcher"),
)


fig = go.Figure(data=data, layout=layout)
fig.show()


In [24]:
import plotly.graph_objects as go 

df_metric = df_long[df_long["Metric"].isin(["L2_PREFETCH Hit", "L2_PREFETCH Miss"])]
total_requests = df_metric.groupby(["Benchmark", "Prefetcher", "Metric"], as_index=False)["Value"].sum()
df_pivot = total_requests.pivot_table(index=["Benchmark", "Prefetcher"], columns="Metric", values="Value", aggfunc="sum").reset_index()
df_pivot['L2_PREFETCH Total'] = df_pivot['L2_PREFETCH Hit'] + df_pivot['L2_PREFETCH Miss']
df_pivot = df_pivot[df_pivot["Prefetcher"] != "no"]

textures = {
    "L2_PREFETCH Hit": ".",  
    "L2_PREFETCH Miss": "x"     
}

prefetcher_colors={
    "ip-stride": "rgb(255, 105, 135)",  
    "next-line": "rgb(135, 206, 235)",  
    "va-ampm-lite": "rgb(186, 140, 186)",
    "adapt-dist": "rgb(255, 165, 0)",
    # "no": "rgb(144, 238, 144)"
}

data = []

for prefetcher in df_pivot["Prefetcher"].unique():
    metric_data = df_pivot[df_pivot["Prefetcher"] == prefetcher]
    
    if metric_data.empty:
        continue
    
    x_data = metric_data["Benchmark"]
    
    y_hits = metric_data["L2_PREFETCH Hit"]
    y_misses = metric_data["L2_PREFETCH Miss"]
    y_data = metric_data["L2_PREFETCH Total"]
    
    data.append(
        go.Bar(
            x=x_data,
            y=y_data,
            name=f"{prefetcher} - Miss",
            hovertext=[f"{prefetcher}<br>Misses: {misses}<br>" 
                   for hits, misses in zip(y_hits, y_misses)],
            marker=dict(
                color=prefetcher_colors.get(prefetcher, "gray"),
                pattern_shape=textures["L2_PREFETCH Miss"]
            ),
            offsetgroup=prefetcher, 
            legendgroup=prefetcher,
            showlegend=False
        )
    )
    
    data.append(
        go.Bar(
            x=x_data,
            y=y_hits,
            hovertext=[f"{prefetcher}<br>Hits: {hits}<br>" 
                   for hits, misses in zip(y_hits, y_misses)],
            marker=dict(
                color=prefetcher_colors.get(prefetcher, "gray"),
                pattern_shape=textures["L2_PREFETCH Hit"]
            ),
            offsetgroup=prefetcher, 
            legendgroup=prefetcher,
            showlegend=False  
        )
    )


for prefetcher, color in prefetcher_colors.items():
    data.append(
        go.Bar(
            x=[None], y=[None],  
            name=f"{prefetcher}", 
            marker=dict(
                color=color,  
                pattern_shape=None  
            ),
            showlegend=True, 
            legendgroup="Prefetcher",  
        )
    )

for metric_type, texture in textures.items():
    metric_type_clean = metric_type.replace("L2_PREFETCH ", "")
    data.append(
        go.Bar(
            x=[None], y=[None],
            name=f"{metric_type_clean}",  
            marker=dict(
                pattern_shape=texture, 
                color='rgba(255,255,255,0)' 
            ),
            showlegend=True,
            legendgroup=metric_type, 
        )
    )

layout = go.Layout(
    title="L2C Prefetch Hit/Miss Prefetch",
    xaxis=dict(title="Benchmark"),
    yaxis=dict(title="Total Requests"),
    barmode="group",  
    legend=dict(title="Prefetcher"),
)


fig = go.Figure(data=data, layout=layout)
fig.show()